# Step 2 — Transform with PySpark
Join and aggregate all data sources using Apache PySpark

In [1]:
!pip install pyspark


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\simeg\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx512m'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_date, avg, count, sum as spark_sum,
    round as spark_round, datediff, when, month, year
)

RAW         = '../data/raw'
PARQUET     = '../data/parquet'
TRANSFORMED = '../data/transformed'
os.makedirs(TRANSFORMED, exist_ok=True)

# Lighter Spark session for local machine
spark = SparkSession.builder \
    .appName('OlistETL') \
    .master('local[2]') \
    .config('spark.driver.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.host', 'localhost') \
    .getOrCreate()


spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)
print('Spark session started!')


Spark version: 3.5.1
Spark session started!


 
 
 ## Load all files into Spark DataFrames

In [2]:
# ============================================================
# OLIST ETL PROJECT - FULL SPARK SETUP
# ============================================================

# -----------------------------
# IMPORTS
# -----------------------------
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    sum as spark_sum,
    round as spark_round,
    datediff,
    when,
    month,
    year
)

# ============================================================
# ENVIRONMENT VARIABLES
# ============================================================

# Java 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"

# Spark path
os.environ["SPARK_HOME"] = r"C:\spark"

# Hadoop path (optional but recommended on Windows)
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Update PATH
os.environ["PATH"] = (
    r"C:\Program Files\Java\jdk-17\bin;"
    r"C:\spark\bin;"
    r"C:\hadoop\bin;"
    + os.environ["PATH"]
)

# ============================================================
# PROJECT PATHS
# ============================================================

RAW = "../data/raw"
PARQUET = "../data/parquet"
TRANSFORMED = "../data/transformed"

# Create folders if missing
os.makedirs(PARQUET, exist_ok=True)
os.makedirs(TRANSFORMED, exist_ok=True)

# ============================================================
# CREATE SPARK SESSION
# ============================================================

spark = SparkSession.builder \
    .appName("OlistETL") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED") \
    .getOrCreate()

# Reduce log noise
spark.sparkContext.setLogLevel("ERROR")

print("=" * 50)
print("Spark Started Successfully!")
print("Spark Version:", spark.version)
print("=" * 50)

# ============================================================
# LOAD CSV FILES
# ============================================================

print("\nLoading datasets...")

orders = spark.read.csv(
    f"{RAW}/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

customers = spark.read.csv(
    f"{RAW}/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

order_items = spark.read.csv(
    f"{RAW}/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

payments = spark.read.csv(
    f"{RAW}/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

reviews = spark.read.csv(
    f"{RAW}/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

products = spark.read.csv(
    f"{RAW}/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

category = spark.read.csv(
    f"{RAW}/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

weather = spark.read.csv(
    f"{PARQUET}/sao_paulo_weather.csv",
    header=True,
    inferSchema=True
)

print("All CSV files loaded successfully!")

# ============================================================
# CREATE SAFE PARQUET FILE
# ============================================================

print("\nCreating Spark-compatible Parquet file...")

orders.write.mode("overwrite").parquet(
    f"{PARQUET}/olist_orders_fixed.parquet"
)

print("Parquet file created!")

# ============================================================
# READ FIXED PARQUET FILE
# ============================================================

orders_parquet = spark.read.parquet(
    f"{PARQUET}/olist_orders_fixed.parquet"
)

print("Parquet file loaded successfully!")

# ============================================================
# SHOW DATA COUNTS
# ============================================================

print("\n" + "=" * 50)
print("DATASET ROW COUNTS")
print("=" * 50)

print(f"Orders:        {orders.count():,}")
print(f"Customers:     {customers.count():,}")
print(f"Order Items:   {order_items.count():,}")
print(f"Payments:      {payments.count():,}")
print(f"Reviews:       {reviews.count():,}")
print(f"Products:      {products.count():,}")
print(f"Categories:    {category.count():,}")
print(f"Weather:       {weather.count():,}")
print(f"Parquet Rows:  {orders_parquet.count():,}")

# ============================================================
# SHOW SCHEMA
# ============================================================

print("\nOrders Schema:")
orders.printSchema()

# ============================================================
# SHOW SAMPLE DATA
# ============================================================

print("\nOrders Sample:")
orders.show(5)

# ============================================================
# SIMPLE TRANSFORMATION
# ============================================================

orders_clean = orders.withColumn(
    "purchase_date",
    to_date(col("order_purchase_timestamp"))
)

print("\nCleaned Orders Sample:")
orders_clean.show(5)

# ============================================================
# CACHE DATAFRAME (optional)
# ============================================================

orders_clean.cache()

# ============================================================
# TEST SPARK JOB
# ============================================================

print("\nRunning Spark job...")

total_orders = orders_clean.count()

print(f"Total Orders: {total_orders:,}")

# ============================================================
# SPARK UI
# ============================================================

print("\nSpark UI:")
print("http://localhost:4040")

print("\nETL Environment Ready!")
print("=" * 50)

# ============================================================
# OPTIONAL STOP
# ============================================================

# spark.stop()

Spark Started Successfully!
Spark Version: 3.5.1

Loading datasets...
All CSV files loaded successfully!

Creating Spark-compatible Parquet file...
Parquet file created!
Parquet file loaded successfully!

DATASET ROW COUNTS
Orders:        99,441
Customers:     99,441
Order Items:   112,650
Payments:      103,886
Reviews:       104,162
Products:      32,951
Categories:    71
Weather:       730
Parquet Rows:  99,441

Orders Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


Orders Sample:
+--------------------+--------------------+------------+------------------------+-------------------+--------------

## Transformation 1 — Clean Orders

In [2]:
# Clean orders — keep only delivered orders, add date columns
orders_clean = orders \
    .filter(col('order_status') == 'delivered') \
    .withColumn('purchase_date', to_date(col('order_purchase_timestamp'))) \
    .withColumn('delivery_date', to_date(col('order_delivered_customer_date'))) \
    .withColumn('estimated_date', to_date(col('order_estimated_delivery_date'))) \
    .withColumn('delivery_days', datediff(col('delivery_date'), col('purchase_date'))) \
    .withColumn('was_late', when(
        col('delivery_date') > col('estimated_date'), 1
    ).otherwise(0)) \
    .withColumn('year',  year(col('purchase_date'))) \
    .withColumn('month', month(col('purchase_date')))

print('Orders after cleaning:', orders_clean.count())
orders_clean.printSchema()

Orders after cleaning: 96478
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- estimated_date: date (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- was_late: integer (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



## Transformation 2 — Join Orders + Customers + Payments + Items

In [3]:
# Join orders with customers
orders_customers = orders_clean.join(
    customers.select('customer_id', 'customer_state', 'customer_city'),
    on='customer_id', how='left'
)

# Aggregate payments per order
payments_agg = payments.groupBy('order_id').agg(
    spark_sum('payment_value').alias('total_payment'),
    count('payment_sequential').alias('payment_count')
)

# Join with payments
orders_payments = orders_customers.join(payments_agg, on='order_id', how='left')

# Aggregate items per order
items_agg = order_items.groupBy('order_id').agg(
    count('order_item_id').alias('item_count'),
    spark_sum('price').alias('items_total'),
    spark_sum('freight_value').alias('freight_total')
)

# Join with items
orders_full = orders_payments.join(items_agg, on='order_id', how='left')

print('Full orders joined:', orders_full.count())
print('Columns:', orders_full.columns)

Full orders joined: 96478
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_date', 'delivery_date', 'estimated_date', 'delivery_days', 'was_late', 'year', 'month', 'customer_state', 'customer_city', 'total_payment', 'payment_count', 'item_count', 'items_total', 'freight_total']


## Transformation 3 — Join with Weather Data

In [4]:
# Clean weather data
weather_clean = weather \
    .withColumn('date', to_date(col('date'))) \
    .withColumn('temp_max', col('temp_max').cast('double')) \
    .withColumn('precipitation', col('precipitation').cast('double'))

# Join orders with weather on purchase date
orders_weather = orders_full.join(
    weather_clean,
    orders_full['purchase_date'] == weather_clean['date'],
    how='left'
).drop('date')

print('Orders with weather:', orders_weather.count())
orders_weather.select('order_id','purchase_date','total_payment','temp_max','precipitation').show(5)

Orders with weather: 96478
+--------------------+-------------+-------------+--------+-------------+
|            order_id|purchase_date|total_payment|temp_max|precipitation|
+--------------------+-------------+-------------+--------+-------------+
|e481f51cbdc54678b...|   2017-10-02|        38.71|    23.0|          5.0|
|53cdb2fc8bc7dce0b...|   2018-07-24|       141.46|    21.1|          0.0|
|47770eb9100c2d0c4...|   2018-08-08|       179.12|    23.0|          0.9|
|949d5b44dbf5de918...|   2017-11-18|         72.2|    27.4|         14.2|
|ad21c59c0840e6cb8...|   2018-02-13|        28.62|    20.2|          3.2|
+--------------------+-------------+-------------+--------+-------------+
only showing top 5 rows



## Transformation 4 — Aggregated Summary Tables

In [5]:
# Table 1: Monthly revenue summary
monthly_revenue = orders_weather.groupBy('year', 'month').agg(
    count('order_id').alias('total_orders'),
    spark_round(spark_sum('total_payment'), 2).alias('total_revenue'),
    spark_round(avg('total_payment'), 2).alias('avg_order_value'),
    spark_round(avg('delivery_days'), 1).alias('avg_delivery_days'),
    spark_sum('was_late').alias('late_deliveries')
).orderBy('year', 'month')

print('Monthly Revenue Table:')
monthly_revenue.show()

Monthly Revenue Table:
+----+-----+------------+-------------+---------------+-----------------+---------------+
|year|month|total_orders|total_revenue|avg_order_value|avg_delivery_days|late_deliveries|
+----+-----+------------+-------------+---------------+-----------------+---------------+
|2016|    9|           1|         NULL|           NULL|             55.0|              1|
|2016|   10|         265|     46566.71|         175.72|             19.6|              2|
|2016|   12|           1|        19.62|          19.62|              5.0|              0|
|2017|    1|         750|    127545.67|         170.06|             12.8|             22|
|2017|    2|        1653|    271298.65|         164.13|             13.3|             49|
|2017|    3|        2546|    414369.39|         162.75|             13.0|            116|
|2017|    4|        2303|    390952.18|         169.76|             15.0|            151|
|2017|    5|        3546|    567066.73|         159.92|             11.4|    

In [6]:
# Table 2: Revenue by state
revenue_by_state = orders_weather.groupBy('customer_state').agg(
    count('order_id').alias('total_orders'),
    spark_round(spark_sum('total_payment'), 2).alias('total_revenue'),
    spark_round(avg('total_payment'), 2).alias('avg_order_value'),
    spark_round(avg('delivery_days'), 1).alias('avg_delivery_days')
).orderBy('total_revenue', ascending=False)

print('Revenue by State:')
revenue_by_state.show()

Revenue by State:
+--------------+------------+-------------+---------------+-----------------+
|customer_state|total_orders|total_revenue|avg_order_value|avg_delivery_days|
+--------------+------------+-------------+---------------+-----------------+
|            SP|       40501|   5770266.19|         142.48|              8.7|
|            RJ|       12350|   2055690.45|         166.45|             15.2|
|            MG|       11354|   1819277.61|         160.23|             11.9|
|            RS|        5345|     861802.4|         161.24|             15.2|
|            PR|        4923|    781919.55|         158.83|             11.9|
|            SC|        3546|     595208.4|         167.85|             14.9|
|            BA|        3256|     591270.6|         181.59|             19.3|
|            DF|        2080|    346146.17|         166.42|             12.9|
|            GO|        1957|    334294.22|         170.82|             15.5|
|            ES|        1995|    317682.65|   

In [7]:
# Table 3: Weather impact on orders
weather_impact = orders_weather.groupBy('precipitation').agg(
    count('order_id').alias('total_orders'),
    spark_round(avg('total_payment'), 2).alias('avg_order_value')
).withColumn('rain_category', when(
    col('precipitation') == 0, 'No Rain'
).when(
    col('precipitation') < 5, 'Light Rain'
).otherwise('Heavy Rain')
).groupBy('rain_category').agg(
    spark_sum('total_orders').alias('total_orders'),
    spark_round(avg('avg_order_value'), 2).alias('avg_order_value')
)

print('Weather Impact on Orders:')
weather_impact.show()

Weather Impact on Orders:
+-------------+------------+---------------+
|rain_category|total_orders|avg_order_value|
+-------------+------------+---------------+
|      No Rain|       33726|         161.81|
|   Heavy Rain|       16667|         159.63|
|   Light Rain|       46085|         158.41|
+-------------+------------+---------------+



## Save Transformed Data

In [8]:
# Save all transformed tables as parquet
orders_weather.write.mode('overwrite').parquet(f'{TRANSFORMED}/orders_full')
monthly_revenue.write.mode('overwrite').parquet(f'{TRANSFORMED}/monthly_revenue')
revenue_by_state.write.mode('overwrite').parquet(f'{TRANSFORMED}/revenue_by_state')
weather_impact.write.mode('overwrite').parquet(f'{TRANSFORMED}/weather_impact')

print('=' * 50)
print('TRANSFORMATION COMPLETE')
print('=' * 50)
print('Saved: orders_full')
print('Saved: monthly_revenue')
print('Saved: revenue_by_state')
print('Saved: weather_impact')
print('Ready for Step 3 — Load!')

TRANSFORMATION COMPLETE
Saved: orders_full
Saved: monthly_revenue
Saved: revenue_by_state
Saved: weather_impact
Ready for Step 3 — Load!


In [ ]:
spark.stop()
print('Spark session stopped.')